# 05. Blocking

When you know in advance a categorical variable that strongly affects the outcome (region, device type, cohort), you can **guarantee** balance on it by randomizing **within** each group. That is blocking: `BlockedCRD` for the design and `BlockedDifferenceInMeans` for the estimator.

## The experiment

Two blocks, `A` and `B`, with very different baselines. The true effect is `0.5` in both. Blocking randomizes within each block, so the treated/control proportion is preserved in each one.

In [ ]:
import numpy as np
import pandas as pd

from skxperiments.core.assignment import BlockedAssignment
from skxperiments.design.blocked_crd import BlockedCRD

rng = np.random.default_rng(5)
n_per = 50
blocks = np.repeat(["A", "B"], n_per)
n = len(blocks)

block_baseline = np.where(blocks == "A", 0.0, 3.0)   # block B starts higher
tau = 0.5
y0 = block_baseline + rng.normal(size=n)
y1 = y0 + tau

df = pd.DataFrame({"block": blocks})
design = BlockedCRD(block_col="block", p=0.5, seed=5)
assignment = design.randomize(df)
t = assignment.data_[assignment.treatment_col_].to_numpy()
data = assignment.data_.copy()
data["y"] = np.where(t == 1, y1, y0)
assignment = BlockedAssignment(
    data=data,
    treatment_col=assignment.treatment_col_,
    design=design,
    block_col=assignment.block_col_,
    block_sizes=assignment.block_sizes_,
    seed=5,
)

## Estimate with a block-size-weighted mean

`BlockedDifferenceInMeans` estimates the effect within each block and combines them by the proportion of units, which is unbiased under blocking.

In [ ]:
from skxperiments.estimators.blocked_difference_in_means import (
    BlockedDifferenceInMeans,
)

est = BlockedDifferenceInMeans(outcome_col="y").fit(assignment)
result = est.estimate()
print(f"ATE (blocked): {result.ate:.3f}  | true: {tau}")
print("ATE per block:", {k: round(v, 3) for k, v in est.block_ates_.items()})

## What we learned

- Blocking **guarantees** balance on the block variable instead of leaving it to chance.
- The estimator combines per-block effects weighted by size, isolating the large baseline difference between `A` and `B`.
- It is the natural choice when a known categorical covariate dominates the outcome's variation.

**Next:** `06. Factorial` estimates main effects and interactions of several factors.